In [ ]:
import io, time, requests, pandas as pd

BASE_URL = ("https://baseballsavant.mlb.com/statcast_search/csv?all=true&type=details"
            "&hfGT=R%7C&hfSea=2026%7C&player_type=pitcher&game_date_gt={d}&game_date_lt={d}")

# Politeness settings: requests go out one at a time, each waiting until at least REQUEST_DELAY
# seconds after the previous one finished. Rate limits / server errors back off 30s, 60s, 120s, 240s.
REQUEST_DELAY = 2
MAX_RETRIES = 4
RETRY_BACKOFF = 30
RETRY_STATUSES = {429, 500, 502, 503, 504}

session = requests.Session()
_last_request = 0.0

def polite_get(url):
    """GET with spacing between requests and exponential backoff on transient errors."""
    global _last_request
    for attempt in range(MAX_RETRIES + 1):
        time.sleep(max(0, REQUEST_DELAY - (time.time() - _last_request)))
        r, error = None, None
        try:
            r = session.get(url, timeout=120)
            if r.status_code in RETRY_STATUSES:
                error = f"HTTP {r.status_code}"
        except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError) as e:
            error = type(e).__name__
        _last_request = time.time()
        if error is None:
            r.raise_for_status()
            return r
        if attempt == MAX_RETRIES:
            raise RuntimeError(f"Giving up after {MAX_RETRIES} retries ({error}): {url}")
        wait = RETRY_BACKOFF * 2 ** attempt
        if r is not None and r.headers.get("Retry-After", "").isdigit():
            wait = max(wait, int(r.headers["Retry-After"]))  # server told us how long to back off
        print(f"    {error} -- waiting {wait}s before retry {attempt + 1}/{MAX_RETRIES}")
        time.sleep(wait)

def fetch_day(d, extra=""):
    """Pull one day of 2026 regular-season Statcast pitch data.
    `extra` is appended to the query string (e.g. an hfABSFlag filter).
    Returns an (empty) DataFrame if there were no matching pitches that day."""
    r = polite_get(BASE_URL.format(d=d) + extra)
    if not r.text.strip():
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(r.text))

In [2]:
# Test with a single day first.
# Note: 2026-03-02 is spring training, and hfGT=R filters to regular-season games
# only, so that date comes back with a header but zero rows. 2026 Opening Day is
# 2026-03-26, so test with a date on/after that.
TEST_DATE = "2026-04-01"
test_df = fetch_day(TEST_DATE)
print(f"{TEST_DATE}: {len(test_df)} rows")
test_df.head()

2026-04-01: 4433 rows


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FF,2026-04-01,92.6,2.12,5.46,"Boyd, Matthew",545361,571510,walk,ball,...,2,1.51,1.25,-1.25,24.2,NaN,NaN,NaN,NaN,NaN
1,SI,2026-04-01,94.6,1.86,6.08,"Sánchez, Cristopher",682928,650911,strikeout,called_strike,...,2,2.28,1.37,1.37,31.0,NaN,NaN,NaN,NaN,NaN
2,FF,2026-04-01,92.0,2.04,5.48,"Boyd, Matthew",545361,571510,NaN,ball,...,2,1.56,1.09,-1.09,24.7,NaN,NaN,NaN,NaN,NaN
3,CH,2026-04-01,80.5,2.35,5.33,"Boyd, Matthew",545361,571510,NaN,foul,...,2,3.33,1.10,-1.10,25.1,26.091181,-40.041316,35.545309,38.491718,47.212774
4,SL,2026-04-01,87.2,1.89,6.14,"Sánchez, Cristopher",682928,650911,NaN,ball,...,2,3.43,-0.02,-0.02,33.1,NaN,NaN,NaN,NaN,NaN


In [3]:
# Full season pull — run this once the single-day test above looks good.
# Starts on Opening Day (2026-03-26) instead of 2026-03-01 so we're not burning
# requests on spring-training dates that will always come back empty.
frames = []
for d in pd.date_range("2026-03-26", "2026-10-01").date:
    df = fetch_day(d)
    if not df.empty:
        frames.append(df)
    print(d, len(df), "rows")
    time.sleep(1)  # be polite to the server

statcast_2026 = pd.concat(frames, ignore_index=True)
statcast_2026.to_parquet("statcast_2026_regular.parquet")
statcast_2026.shape

2026-03-26 3310 rows
2026-03-27 2247 rows
2026-03-28 4775 rows
2026-03-29 3662 rows
2026-03-30 4346 rows
2026-03-31 4012 rows
2026-04-01 4433 rows
2026-04-02 923 rows
2026-04-03 4156 rows
2026-04-04 4622 rows
2026-04-05 5052 rows
2026-04-06 3800 rows
2026-04-07 4516 rows
2026-04-08 4483 rows
2026-04-09 1766 rows
2026-04-10 4313 rows
2026-04-11 4602 rows
2026-04-12 4376 rows
2026-04-13 3039 rows
2026-04-14 4390 rows
2026-04-15 4343 rows
2026-04-16 3100 rows
2026-04-17 4520 rows
2026-04-18 4713 rows
2026-04-19 4513 rows
2026-04-20 3070 rows
2026-04-21 4719 rows
2026-04-22 4280 rows
2026-04-23 2599 rows
2026-04-24 4092 rows
2026-04-25 4397 rows
2026-04-26 4606 rows
2026-04-27 2407 rows
2026-04-28 4324 rows
2026-04-29 3890 rows
2026-04-30 3289 rows
2026-05-01 4530 rows
2026-05-02 4359 rows
2026-05-03 4241 rows
2026-05-04 3262 rows
2026-05-05 3681 rows
2026-05-06 4324 rows
2026-05-07 2977 rows
2026-05-08 4327 rows
2026-05-09 4248 rows
2026-05-10 4276 rows
2026-05-11 1772 rows
2026-05-12 448

(716534, 119)

In [ ]:
# ABS challenge filters.
# The site's JS escapes every "." as "\." before sending the filter, so the raw value
# "is..challengeabs..review" has to go out as "is\.\.challengeabs\.\.review|".
# Without that escaping the server silently ignores the filter and returns every pitch.
from urllib.parse import quote

ABS_FLAGS = {
    "abs_flag":             "is..challengeabs..review",
    "challenge_overturned": "is..challengeabs..overturned",
    "challengeable_pitch":  "is..challengeabs..opp",
    "by_batter":            "is..challengeabs..review..bat",
    "by_pitcher":           "is..challengeabs..review..pit",
    "by_catcher":           "is..challengeabs..review..cat",
    "by_fielder":           "is..challengeabs..review..fld",
}
FLAG_COLS = ["abs_flag", "challenge_overturned", "challengeable_pitch"]
BY_COLS = {"by_batter": "batter", "by_pitcher": "pitcher", "by_catcher": "catcher", "by_fielder": "fielder"}

PITCH_KEY = ["game_pk", "at_bat_number", "pitch_number"]  # unique per pitch

def abs_filter(value):
    return "&hfABSFlag=" + quote(value.replace(".", "\\.") + "|", safe=".")

def fetch_abs_flags(d):
    """One row per pitch on date d that matches any ABS filter, with a 0/1 column per filter.
    Request spacing is handled inside fetch_day."""
    parts = []
    for col, value in ABS_FLAGS.items():
        df = fetch_day(d, abs_filter(value))
        if len(df):
            parts.append(df[PITCH_KEY].assign(flag=col))
    if not parts:
        return pd.DataFrame({c: pd.Series(dtype="int64") for c in PITCH_KEY + list(ABS_FLAGS)})
    flags = (pd.concat(parts).assign(v=1)
               .pivot_table(index=PITCH_KEY, columns="flag", values="v", aggfunc="max", fill_value=0)
               .reindex(columns=list(ABS_FLAGS), fill_value=0)
               .reset_index())
    flags.columns.name = None
    return flags

def add_abs_columns(pitches, flags):
    """Join ABS flags onto the pitch data -> abs_flag, challenge_overturned, challengeable_pitch, challenger."""
    out = pitches.merge(flags, on=PITCH_KEY, how="left", validate="one_to_one")
    cols = FLAG_COLS + list(BY_COLS)
    out[cols] = out[cols].fillna(0).astype(int)
    assert (out[["by_batter", "by_pitcher", "by_catcher"]].sum(axis=1) <= 1).all(), "pitch has >1 challenger"
    # "Challenged by Fielder" = pitcher OR catcher, so it only becomes the label as a fallback
    # (flagged fielder but neither pitcher nor catcher). Unchallenged pitches stay NaN.
    out["challenger"] = pd.Series(pd.NA, index=out.index, dtype="object")
    for col, name in reversed(BY_COLS.items()):  # lowest priority first; later assignments win
        out.loc[out[col] == 1, "challenger"] = name
    out["challenger"] = pd.Categorical(out["challenger"], categories=list(BY_COLS.values()))
    return out.drop(columns=list(BY_COLS))

In [ ]:
# All 4 ABS columns for a single date (uses TEST_DATE / test_df from the test cell above).
flags = fetch_abs_flags(TEST_DATE)
test_abs = add_abs_columns(test_df, flags)

# Sanity checks
assert (flags[list(ABS_FLAGS)].sum() < len(test_df)).all(), "an hfABSFlag filter was ignored -- check the escaping"
assert len(test_abs) == len(test_df)                                          # join didn't add/drop rows
assert len(flags.merge(test_df[PITCH_KEY], on=PITCH_KEY)) == len(flags)       # every flagged pitch matched
assert (test_abs["challenge_overturned"] <= test_abs["abs_flag"]).all()       # overturned => challenged
assert (test_abs["abs_flag"] <= test_abs["challengeable_pitch"]).all()        # challenged => challengeable
assert (test_abs["challenger"].notna() == (test_abs["abs_flag"] == 1)).all()  # challenger set iff challenged

print(f"{TEST_DATE}: {len(test_abs)} pitches | {test_abs['challengeable_pitch'].sum()} challengeable | "
      f"{test_abs['abs_flag'].sum()} challenged | {test_abs['challenge_overturned'].sum()} overturned")
pd.crosstab(test_abs["challenger"], test_abs["challenge_overturned"], margins=True)

## Full 2026 regular season: all pitches + ABS columns

Pulls every regular-season pitch with the 4 ABS columns (`abs_flag`, `challenge_overturned`, `challengeable_pitch`, `challenger`) one day at a time, using the helpers above. Run the setup cell and the ABS helpers cell first.

- **Server load:** requests go out strictly one at a time, at least `REQUEST_DELAY` (2s) apart, with exponential backoff on rate limits and errors. A game day takes 8 requests (1 for all pitches + 7 ABS filters), and off days take 1. That's about 1,470 requests in total, so expect roughly **1 to 1.5 hours**.
- **Resumable:** each finished day is saved to `daily_cache/`. If the run is interrupted, re-run the pull cell, and finished days are skipped without requesting them again. Delete `daily_cache/` to force a completely fresh pull.
- **Output:** `statcast_2026_regular_abs.parquet` next to this notebook. Set `OUTPUT_FORMAT = "csv"` to write a CSV instead.

In [ ]:
from pathlib import Path

SEASON_START, SEASON_END = "2026-03-26", "2026-09-27"  # 2026 regular season: Opening Day -> final day
OUTPUT_FORMAT = "parquet"                             # "parquet" or "csv"
OUTPUT_NAME = "statcast_2026_regular_abs"             # saved next to this notebook
CACHE_DIR = Path("daily_cache")                       # one file per finished day, so reruns resume
SAVANT_ROW_CAP = 25_000                               # Savant's CSV export silently truncates here

def pull_day(d):
    """All pitches on date d with the 4 ABS columns, plus a dict of data-quality checks."""
    pitches = fetch_day(d)
    if pitches.empty:  # off day -- skip the 7 ABS requests
        return pitches, {"date": d, "pitches": 0}
    if len(pitches) >= SAVANT_ROW_CAP:
        raise RuntimeError(f"{d}: hit Savant's {SAVANT_ROW_CAP:,}-row cap, so the data is truncated")
    flags = fetch_abs_flags(d)
    if (flags[list(ABS_FLAGS)].sum() >= len(pitches)).any():
        raise RuntimeError(f"{d}: an hfABSFlag filter was ignored -- check the escaping")
    day = add_abs_columns(pitches, flags)
    return day, {
        "date": d,
        "pitches": len(day),
        "challengeable": int(day["challengeable_pitch"].sum()),
        "challenged": int(day["abs_flag"].sum()),
        "overturned": int(day["challenge_overturned"].sum()),
        # Problem counters: all 0 on the 2026-04-01 test, so anything nonzero is worth a look.
        "unmatched_flags": len(flags) - len(flags.merge(pitches[PITCH_KEY], on=PITCH_KEY)),
        "overturned_not_challenged": int(((day["challenge_overturned"] == 1) & (day["abs_flag"] == 0)).sum()),
        "challenged_not_challengeable": int(((day["abs_flag"] == 1) & (day["challengeable_pitch"] == 0)).sum()),
        "challenger_mismatch": int((day["challenger"].notna() != (day["abs_flag"] == 1)).sum()),
    }

def save_output(df, name, fmt):
    if fmt == "parquet":
        path = f"{name}.parquet"
        df.to_parquet(path, index=False)
    elif fmt == "csv":
        path = f"{name}.csv"
        df.to_csv(path, index=False, encoding="utf-8-sig")  # BOM so Excel shows accented names correctly
    else:
        raise ValueError(f"OUTPUT_FORMAT must be 'parquet' or 'csv', got {fmt!r}")
    return path

In [ ]:
# Pull the season one day at a time. Safe to interrupt and re-run: finished days are skipped.
CACHE_DIR.mkdir(exist_ok=True)
checks_path = CACHE_DIR / "daily_checks.csv"
checks = pd.read_csv(checks_path).to_dict("records") if checks_path.exists() else []
done = {c["date"] for c in checks}

dates = [str(d) for d in pd.date_range(SEASON_START, SEASON_END).date]
print(f"{len(done)} of {len(dates)} days already cached")
for i, d in enumerate(dates, 1):
    if d in done:
        continue
    day, chk = pull_day(d)
    if not day.empty:
        day.to_parquet(CACHE_DIR / f"{d}.parquet", index=False)
    checks.append(chk)
    pd.DataFrame(checks).to_csv(checks_path, index=False)  # written after the day file, so a crash just re-pulls that day
    print(f"[{i}/{len(dates)}] {d}: {chk['pitches']:,} pitches, {chk.get('challenged', 0)} challenged")

In [ ]:
# Combine the cached days, run season-level checks, and save.
checks_df = pd.DataFrame(checks)
checks_df = checks_df[checks_df["date"].isin(dates)]
missing = sorted(set(dates) - set(checks_df["date"]))
assert not missing, f"{len(missing)} days not pulled yet (e.g. {missing[:3]}) -- re-run the pull cell"

season = pd.concat([pd.read_parquet(CACHE_DIR / f"{d}.parquet") for d in dates
                    if (CACHE_DIR / f"{d}.parquet").exists()], ignore_index=True)
season["challenger"] = pd.Categorical(season["challenger"], categories=list(BY_COLS.values()))

assert len(season) == checks_df["pitches"].sum()                 # every cached day made it in
assert not season.duplicated(PITCH_KEY).any(), "duplicate pitches across days"

problem_cols = ["unmatched_flags", "overturned_not_challenged", "challenged_not_challengeable", "challenger_mismatch"]
problems = checks_df[checks_df.reindex(columns=problem_cols).fillna(0).sum(axis=1) > 0]
print("Days with 0 pitches:", checks_df.loc[checks_df["pitches"] == 0, "date"].tolist())
print("Days with data-quality flags:", "none" if problems.empty else f"\n{problems.to_string(index=False)}")

path = save_output(season, OUTPUT_NAME, OUTPUT_FORMAT)
print(f"Saved {len(season):,} pitches x {season.shape[1]} columns to {path}")
print(f"{season['challengeable_pitch'].sum():,} challengeable | {season['abs_flag'].sum():,} challenged | "
      f"{season['challenge_overturned'].sum():,} overturned")
pd.crosstab(season["challenger"], season["challenge_overturned"], margins=True)